# PidginMed-ASR — Google MedASR Reviewer-Updated Pipeline

This notebook follows the same reviewer-oriented structure as the corrected Whisper notebook while preserving the Google MedASR CTC workflow.

It includes:

- one shared speaker-independent split manifest;
- automatic recovery from an outdated split manifest;
- lazy audio decoding with `librosa` instead of `datasets.Audio` casting;
- stable sample IDs for strict cross-model alignment;
- zero-shot and fine-tuned evaluation;
- architecture-specific hyperparameter trials;
- epoch-wise training loss, validation loss, WER, and CER;
- medical terminology metrics;
- speaker-level and code-switch-level evaluation;
- reviewer-ready CSV and JSON outputs.

Run the notebook from top to bottom.

In [2]:
%pip install -q -U transformers datasets evaluate jiwer accelerate soundfile librosa scikit-learn tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 144.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 145.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but yo

In [19]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mon Oct 12 01:59:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       3MiB /  23034MiB |      0%      Default |
|          

## 1. Configuration and imports

In [20]:
from __future__ import annotations

import ast
import gc
import hashlib
import json
import os
import random
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Union

import librosa
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from jiwer import cer, wer
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCTC,
    AutoProcessor,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "google/medasr"

BASE_DIR = Path("/content/drive/MyDrive/Brock_University/pidgindataset2026October/")
METADATA_PATH = BASE_DIR / "root/Metadata.csv"
AUDIO_ROOT = BASE_DIR
OUTPUT_ROOT = BASE_DIR /"medasr"
SPLIT_MANIFEST_PATH = (BASE_DIR / "shared_split_manifest.csv")

print(AUDIO_ROOT)
print(METADATA_PATH)
print(SPLIT_MANIFEST_PATH)

VALIDATED_LEXICON_PATH = None

TEST_SIZE = 0.20
VAL_SIZE_WITHIN_REMAINING = 0.15

RUN_ZERO_SHOT = True
RUN_TRAINING = True

ZERO_SHOT_BATCH_SIZE = 16
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
EFFECTIVE_BATCH_SIZE = 32

LEARNING_RATES = [1e-5]
EPOCH_OPTIONS = [5]
WEIGHT_DECAYS = [0.01]

EARLY_STOPPING_PATIENCE = 2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Metadata:", METADATA_PATH)
print("Output:", OUTPUT_ROOT)

/content/drive/MyDrive/Brock_University/pidgindataset2026October
/content/drive/MyDrive/Brock_University/pidgindataset2026October/root/Metadata.csv
/content/drive/MyDrive/Brock_University/pidgindataset2026October/shared_split_manifest.csv
Device: cuda
Metadata: /content/drive/MyDrive/Brock_University/pidgindataset2026October/root/Metadata.csv
Output: /content/drive/MyDrive/Brock_University/pidgindataset2026October/medasr


In [21]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

## 2. Metadata loading and normalization

In [22]:
def normalize_text(text: Any) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z\s']", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_terms(value: Any) -> List[str]:
    if isinstance(value, list):
        raw_terms = value
    elif value is None or (
        not isinstance(value, (list, tuple, set))
        and pd.isna(value)
    ):
        raw_terms = []
    else:
        try:
            parsed = ast.literal_eval(str(value))
            raw_terms = parsed if isinstance(parsed, list) else [parsed]
        except Exception:
            raw_terms = [
                item.strip()
                for item in str(value).split(",")
                if item.strip()
            ]

    cleaned = [
        normalize_text(term)
        for term in raw_terms
        if normalize_text(term)
    ]

    return list(dict.fromkeys(cleaned))


def make_sample_id(row: pd.Series) -> str:
    for column in ("sample_id", "utterance_id", "id"):
        if column in row and pd.notna(row[column]):
            return str(row[column])

    identity = "|".join(
        [
            str(row.get("speaker_id", "")),
            Path(str(row.get("audio_path", ""))).name,
            normalize_text(row.get("pidgin_transcription", "")),
        ]
    )

    return hashlib.sha1(
        identity.encode("utf-8")
    ).hexdigest()[:20]

In [23]:
df = pd.read_csv(METADATA_PATH)
df.columns = df.columns.str.lower().str.strip()

required_columns = [
    "audio_path",
    "pidgin_transcription",
    "speaker_id",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Metadata is missing required columns: {missing_columns}"
    )

df = df.dropna(
    subset=required_columns
).copy()

if "audio_length" in df.columns:
    df["audio_length"] = pd.to_numeric(
        df["audio_length"]
        .astype(str)
        .str.replace("secs", "", regex=False),
        errors="coerce",
    )
    df = df[
        df["audio_length"].fillna(0) > 2
    ].copy()

df["speaker_id"] = df["speaker_id"].astype(str)
df["sample_id"] = df.apply(make_sample_id, axis=1)

if df["sample_id"].duplicated().any():
    duplicates = (
        df.loc[
            df["sample_id"].duplicated(keep=False),
            ["sample_id", "audio_path", "speaker_id"],
        ]
        .head(10)
        .to_dict("records")
    )
    raise ValueError(
        "Duplicate sample IDs detected. Examples: "
        f"{duplicates}"
    )

def resolve_audio_path(path_value: Any) -> str:
    path = Path(str(path_value))

    if not path.is_absolute():
        path = AUDIO_ROOT / path

    return str(path.resolve())


df["audio"] = df["audio_path"].map(resolve_audio_path)
df["reference"] = df["pidgin_transcription"].map(
    normalize_text
)

medical_source = None
for candidate in (
    "medical_terms_clean",
    "medical_terms_pidgen",
):
    if candidate in df.columns:
        medical_source = candidate
        break

if medical_source is None:
    df["medical_terms_pidgen"] = [[] for _ in range(len(df))]
else:
    df["medical_terms_pidgen"] = df[medical_source].map(
        parse_terms
    )

if VALIDATED_LEXICON_PATH is not None:
    lexicon = pd.read_csv(VALIDATED_LEXICON_PATH)
    lexicon["sample_id"] = lexicon["sample_id"].astype(str)
    lexicon["medical_terms_pidgen"] = lexicon[
        "medical_terms_pidgen"
    ].map(parse_terms)

    df = (
        df.drop(columns=["medical_terms_pidgen"])
        .merge(
            lexicon[["sample_id", "medical_terms_pidgen"]],
            on="sample_id",
            how="left",
            validate="one_to_one",
        )
    )

    df["medical_terms_pidgen"] = df["medical_terms_pidgen"].map(
        parse_terms
    )

df = df.reset_index(drop=True)

print("Samples:", len(df))
print("Speakers:", df["speaker_id"].nunique())
display(df.head())

Samples: 10218
Speakers: 21


,speaker_id,audio_path,pidgin_transcription,english_translation,medical_terms_english,medical_terms_pidgen,audio_length,sample_rate,num_channels,speech_rate,signal_to_noise_ratio,silence_ratio,audio,file_exists,text,reference,sample_id
0,Speaker_1,root/audio/Speaker_1/Speaker1-1.wav,e depend on your effort how you take dey do ex...,it all depends on your efforts regarding regul...,"['care', 'knee', 'walk', 'two days', 'one to t...","[exercise, everytime, after two day, normally,...",19.04,16000.0,1.0,2.678378,54.947548,0.319377,/content/drive/MyDrive/Brock_University/pidgin...,True,e depend on your effort how you take dey do ex...,e depend on your effort how you take dey do ex...,6a840a4ad1fd60283227
1,Speaker_1,root/audio/Speaker_1/Speaker1-2.wav,my daughter make four years two nights ago she...,my daughter is four years old two nights ago s...,"['daughter', 'four years old', 'vomited', 'sor...","[daughter, wake up, vomited, pain, hot, parace...",34.48,16000.0,1.0,2.580879,53.953575,0.334655,/content/drive/MyDrive/Brock_University/pidgin...,True,my daughter make four years two nights ago she...,my daughter make four years two nights ago she...,fa777d58c8f323da62e2
2,Speaker_1,root/audio/Speaker_1/Speaker1-3.wav,hi dis na retention cyst e go need excise for ...,hi this is definitely a retention cyst it need...,"['retention cyst', 'mass', 'histopathological ...","[reasons, removed, important part of the body,...",21.97,16000.0,1.0,2.321869,57.647476,0.335672,/content/drive/MyDrive/Brock_University/pidgin...,True,hi dis na retention cyst e go need excise for ...,hi dis na retention cyst e go need excise for ...,01fceaa9e74b77698df4
3,Speaker_1,root/audio/Speaker_1/Speaker1-4.wav,i be thirty one years woman and i get blod clo...,i am a thirty one year old woman with a blood ...,"['thirty one year old woman', 'head injury', '...","[thirty one years woman, not, head injury, inj...",34.61,16000.0,1.0,2.571562,49.890247,0.308394,/content/drive/MyDrive/Brock_University/pidgin...,True,i be thirty one years woman and i get blod clo...,i be thirty one years woman and i get blod clo...,3d812380dee30f23b366
4,Speaker_1,root/audio/Speaker_1/Speaker1-5.wav,i be forty five years old man my platelet coun...,i am forty five years old male my platelet cou...,"['forty five years old male', 'platelet count'...","[forty five years old man, platelet count, thi...",42.63,16000.0,1.0,2.040783,50.834831,0.307166,/content/drive/MyDrive/Brock_University/pidgin...,True,i be forty five years old man my platelet coun...,i be forty five years old man my platelet coun...,ac3bdc92665cb5600166


## 3. Shared speaker-independent split

In [24]:
def create_or_load_speaker_split(
    frame: pd.DataFrame,
    manifest_path: Union[str, Path],
    rebuild_if_incomplete: bool = True,
):
    frame = frame.copy()
    manifest_path = Path(manifest_path)

    frame["sample_id"] = frame["sample_id"].astype(str)
    frame["speaker_id"] = frame["speaker_id"].astype(str)

    def create_new_manifest(current_frame):
        speakers = sorted(
            current_frame["speaker_id"].unique().tolist()
        )

        train_speakers, test_speakers = train_test_split(
            speakers,
            test_size=TEST_SIZE,
            random_state=SEED,
        )

        train_speakers, validation_speakers = (
            train_test_split(
                train_speakers,
                test_size=VAL_SIZE_WITHIN_REMAINING,
                random_state=SEED,
            )
        )

        split_map = {
            **{
                speaker: "train"
                for speaker in train_speakers
            },
            **{
                speaker: "validation"
                for speaker in validation_speakers
            },
            **{
                speaker: "test"
                for speaker in test_speakers
            },
        }

        current_frame["split"] = (
            current_frame["speaker_id"]
            .map(split_map)
        )

        if current_frame["split"].isna().any():
            raise ValueError(
                "At least one speaker could not be assigned "
                "to a split."
            )

        manifest_columns = [
            "sample_id",
            "speaker_id",
            "split",
            "audio_path",
            "pidgin_transcription",
        ]

        if "english_translation" in current_frame.columns:
            manifest_columns.append("english_translation")

        for optional_column in (
            "audio_length",
            "code_switch_category",
            "cmi",
            "english_ratio",
            "pidgin_ratio",
        ):
            if optional_column in current_frame.columns:
                manifest_columns.append(optional_column)

        manifest = current_frame[
            manifest_columns
        ].copy()

        manifest_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )
        manifest.to_csv(
            manifest_path,
            index=False,
        )

        print(
            "Created shared split manifest:",
            manifest_path,
        )
        return current_frame, manifest

    if not manifest_path.exists():
        return create_new_manifest(frame)

    manifest = pd.read_csv(
        manifest_path,
        dtype={
            "sample_id": str,
            "speaker_id": str,
        },
    )

    required = {
        "sample_id",
        "speaker_id",
        "split",
    }

    if not required.issubset(manifest.columns):
        if rebuild_if_incomplete:
            print(
                "Existing manifest is missing required "
                "columns. Rebuilding."
            )
            return create_new_manifest(frame)

        raise ValueError(
            f"Manifest must contain: {sorted(required)}"
        )

    current_ids = set(frame["sample_id"])
    manifest_ids = set(manifest["sample_id"])

    missing_ids = current_ids - manifest_ids

    if missing_ids:
        if not rebuild_if_incomplete:
            raise ValueError(
                "Existing split manifest is incomplete. "
                f"Examples: {list(missing_ids)[:10]}"
            )

        backup_path = manifest_path.with_name(
            manifest_path.stem
            + "_old_"
            + time.strftime("%Y%m%d_%H%M%S")
            + manifest_path.suffix
        )

        manifest.to_csv(
            backup_path,
            index=False,
        )

        print(
            f"Manifest is missing {len(missing_ids)} "
            "current samples."
        )
        print(
            "Saved previous manifest as:",
            backup_path,
        )

        return create_new_manifest(frame)

    merged = frame.merge(
        manifest[
            ["sample_id", "speaker_id", "split"]
        ],
        on="sample_id",
        how="left",
        suffixes=("", "_manifest"),
        validate="one_to_one",
    )

    mismatch = (
        merged["speaker_id"]
        != merged["speaker_id_manifest"]
    )

    if mismatch.any():
        examples = merged.loc[
            mismatch,
            [
                "sample_id",
                "speaker_id",
                "speaker_id_manifest",
            ],
        ].head(10)

        raise ValueError(
            "Speaker IDs disagree with the manifest:\n"
            f"{examples}"
        )

    merged = merged.drop(
        columns=["speaker_id_manifest"]
    )

    print(
        "Loaded shared split manifest:",
        manifest_path,
    )

    return merged, manifest


df, split_manifest = create_or_load_speaker_split(
    df,
    SPLIT_MANIFEST_PATH,
)

train_df = (
    df[df["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)
val_df = (
    df[df["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)
test_df = (
    df[df["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

# Remove empty medical-term rows from TEST SET ONLY
original_test_count = len(test_df)

def has_medical_terms(value):
    if value is None:
        return False

    if isinstance(value, (list, tuple, set)):
        return len(value) > 0

    if isinstance(value, str):
        return value.strip().lower() not in {
            "", "[]", "nan", "none", "null"
        }

    return not pd.isna(value)


test_df = (
    test_df[
        test_df["medical_terms_pidgen"].apply(has_medical_terms)
    ]
    .copy()
    .reset_index(drop=True)
)

print(f"Original test samples: {original_test_count}")
print(f"Filtered test samples: {len(test_df)}")
print(f"Removed samples: {original_test_count - len(test_df)}")

train_speakers = set(train_df["speaker_id"])
val_speakers = set(val_df["speaker_id"])
test_speakers = set(test_df["speaker_id"])

assert train_speakers.isdisjoint(val_speakers)
assert train_speakers.isdisjoint(test_speakers)
assert val_speakers.isdisjoint(test_speakers)

summary_aggregations = {
    "samples": ("sample_id", "count"),
    "speakers": ("speaker_id", "nunique"),
}

if "audio_length" in df.columns:
    summary_aggregations["hours"] = (
        "audio_length",
        lambda values: values.sum() / 3600,
    )

split_summary = (
    df.groupby("split")
    .agg(**summary_aggregations)
    .reset_index()
)

display(split_summary)
print("Train speakers:", sorted(train_speakers))
print("Validation speakers:", sorted(val_speakers))
print("Test speakers:", sorted(test_speakers))

Loaded shared split manifest: /content/drive/MyDrive/Brock_University/pidgindataset2026October/shared_split_manifest.csv
Original test samples: 2245
Filtered test samples: 2223
Removed samples: 22


,split,samples,speakers,hours
0,test,2245,5,17.761647
1,train,6474,13,48.963886
2,validation,1499,3,12.255000


Train speakers: ['Speaker_11', 'Speaker_12', 'Speaker_13', 'Speaker_15', 'Speaker_16', 'Speaker_18', 'Speaker_19', 'Speaker_20', 'Speaker_3', 'Speaker_5', 'Speaker_7', 'Speaker_8', 'Speaker_9']
Validation speakers: ['Speaker_14', 'Speaker_2', 'Speaker_21']
Test speakers: ['Speaker_1', 'Speaker_10', 'Speaker_17', 'Speaker_4', 'Speaker_6']


## 4. Lazy audio loading and Hugging Face datasets

In [25]:
def dataframe_to_dataset(
    frame: pd.DataFrame,
) -> Dataset:
    work = frame.copy()
    work["audio"] = work["audio"].astype(str)

    return Dataset.from_pandas(
        work,
        preserve_index=False,
    )


def load_audio_16k(
    audio_value: Any,
) -> Dict[str, Any]:
    if isinstance(audio_value, dict) and "array" in audio_value:
        waveform = np.asarray(
            audio_value["array"],
            dtype=np.float32,
        )
        sampling_rate = int(
            audio_value.get("sampling_rate", 16000)
        )

        if sampling_rate != 16000:
            waveform = librosa.resample(
                waveform,
                orig_sr=sampling_rate,
                target_sr=16000,
            )
            sampling_rate = 16000

        return {
            "array": waveform,
            "sampling_rate": sampling_rate,
        }

    audio_path = Path(str(audio_value))

    if not audio_path.exists():
        raise FileNotFoundError(
            f"Audio file not found: {audio_path}"
        )

    waveform, sampling_rate = librosa.load(
        audio_path,
        sr=16000,
        mono=True,
    )

    return {
        "array": np.asarray(
            waveform,
            dtype=np.float32,
        ),
        "sampling_rate": int(sampling_rate),
    }


train_raw_ds = dataframe_to_dataset(train_df)
val_raw_ds = dataframe_to_dataset(val_df)
test_raw_ds = dataframe_to_dataset(test_df)

sample_audio = load_audio_16k(
    train_raw_ds[0]["audio"]
)

print(train_raw_ds)
print(val_raw_ds)
print(test_raw_ds)
print(
    "Sanity check:",
    sample_audio["sampling_rate"],
    len(sample_audio["array"]),
)

Dataset({
    features: ['speaker_id', 'audio_path', 'pidgin_transcription', 'english_translation', 'medical_terms_english', 'medical_terms_pidgen', 'audio_length', 'sample_rate', 'num_channels', 'speech_rate', 'signal_to_noise_ratio', 'silence_ratio', 'audio', 'file_exists', 'text', 'reference', 'sample_id', 'split'],
    num_rows: 6474
})
Dataset({
    features: ['speaker_id', 'audio_path', 'pidgin_transcription', 'english_translation', 'medical_terms_english', 'medical_terms_pidgen', 'audio_length', 'sample_rate', 'num_channels', 'speech_rate', 'signal_to_noise_ratio', 'silence_ratio', 'audio', 'file_exists', 'text', 'reference', 'sample_id', 'split'],
    num_rows: 1499
})
Dataset({
    features: ['speaker_id', 'audio_path', 'pidgin_transcription', 'english_translation', 'medical_terms_english', 'medical_terms_pidgen', 'audio_length', 'sample_rate', 'num_channels', 'speech_rate', 'signal_to_noise_ratio', 'silence_ratio', 'audio', 'file_exists', 'text', 'reference', 'sample_id', 'sp

## 5. Evaluation metrics

In [26]:
def extract_terms(
    text: str,
    lexicon_terms: Sequence[str],
) -> set:
    normalized = normalize_text(text)
    token_set = set(normalized.split())
    padded = f" {normalized} "

    found = set()

    for term in map(normalize_text, lexicon_terms):
        if not term:
            continue

        if " " in term:
            if f" {term} " in padded:
                found.add(term)
        elif term in token_set:
            found.add(term)

    return found


def compute_row_metrics(
    reference: str,
    prediction: str,
    lexicon_terms: Sequence[str],
) -> Dict[str, Any]:
    reference_terms = extract_terms(
        reference,
        lexicon_terms,
    )
    predicted_terms = extract_terms(
        prediction,
        lexicon_terms,
    )

    return {
        "wer": wer(reference, prediction),
        "cer": cer(reference, prediction),
        "medical_tp": len(
            reference_terms & predicted_terms
        ),
        "medical_fp": len(
            predicted_terms - reference_terms
        ),
        "medical_fn": len(
            reference_terms - predicted_terms
        ),
        "medical_exact": int(
            reference_terms == predicted_terms
        ),
        "reference_medical_terms": sorted(
            reference_terms
        ),
        "predicted_medical_terms": sorted(
            predicted_terms
        ),
        "missing_medical_terms": sorted(
            reference_terms - predicted_terms
        ),
        "spurious_medical_terms": sorted(
            predicted_terms - reference_terms
        ),
    }


def compute_overall_metrics(
    predictions: pd.DataFrame,
) -> Dict[str, Any]:
    tp = int(predictions["medical_tp"].sum())
    fp = int(predictions["medical_fp"].sum())
    fn = int(predictions["medical_fn"].sum())

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    medical_references = []
    medical_predictions = []

    for _, row in predictions.iterrows():
        if row["reference_medical_terms"]:
            medical_references.append(
                " ".join(row["reference_medical_terms"])
            )
            medical_predictions.append(
                " ".join(row["predicted_medical_terms"])
            )

    return {
        "wer": wer(
            predictions["reference"].tolist(),
            predictions["prediction"].tolist(),
        ),
        "cer": cer(
            predictions["reference"].tolist(),
            predictions["prediction"].tolist(),
        ),
        "mwer": (
            wer(
                medical_references,
                medical_predictions,
            )
            if medical_references
            else np.nan
        ),
        "medical_precision": precision,
        "medical_recall": recall,
        "medical_f1": f1,
        "medical_accuracy": float(
            predictions["medical_exact"].mean()
        ),
        "medical_tp": tp,
        "medical_fp": fp,
        "medical_fn": fn,
        "n": len(predictions),
    }

In [27]:
!pip install -q -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 48.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.


In [28]:
from huggingface_hub import notebook_login

notebook_login()

## 6. Load Google MedASR

In [29]:
processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

zero_shot_model = (
    AutoModelForCTC
    .from_pretrained(MODEL_NAME)
    .float()
    .to(DEVICE)
)

print("Processor:", type(processor).__name__)
print("Model:", type(zero_shot_model).__name__)

processor_config.json:   0%|          | 0.00/371 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/284 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  247kB            

...

added_tokens.json:   0%|          | 0.00/2.41k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  421MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/368 [00:00<?, ?it/s]

Processor: LasrProcessor
Model: LasrForCTC


## 7. Zero-shot evaluation

In [30]:
def get_model_input_key(
    processed_inputs: Any,
) -> str:
    if "input_features" in processed_inputs:
        return "input_features"

    if "input_values" in processed_inputs:
        return "input_values"

    raise KeyError(
        "Processor output contains neither "
        "`input_features` nor `input_values`."
    )

In [31]:
def transcribe_dataset(
    dataset: Dataset,
    source_frame: pd.DataFrame,
    model,
    batch_size: int = 1,
    description: str = "Google MedASR inference",
) -> pd.DataFrame:
    model.eval()
    predictions = []

    progress = tqdm(
        range(0, len(dataset), batch_size),
        desc=description,
        unit="batch",
        dynamic_ncols=True,
    )

    for start in progress:
        stop = min(start + batch_size, len(dataset))

        waveforms = [
            load_audio_16k(
                dataset[index]["audio"]
            )["array"]
            for index in range(start, stop)
        ]

        model_inputs = processor(
            waveforms,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True,
        )

        input_key = get_model_input_key(
            model_inputs
        )

        model_kwargs = {
            input_key: model_inputs[input_key].to(
                DEVICE
            )
        }

        if "attention_mask" in model_inputs:
            model_kwargs["attention_mask"] = (
                model_inputs["attention_mask"].to(
                    DEVICE
                )
            )

        with torch.no_grad():
            logits = model(
                **model_kwargs
            ).logits

        predicted_ids = torch.argmax(
            logits,
            dim=-1,
        )

        decoded = processor.batch_decode(
            predicted_ids
        )

        predictions.extend(
            normalize_text(text)
            for text in decoded
        )

        if predictions:
            references = source_frame[
                "reference"
            ].iloc[: len(predictions)].tolist()

            progress.set_postfix(
                WER=f"{wer(references, predictions):.4f}"
            )

    output = source_frame.copy()
    output["prediction"] = predictions

    row_metrics = output.apply(
        lambda row: pd.Series(
            compute_row_metrics(
                row["reference"],
                row["prediction"],
                row["medical_terms_pidgen"],
            )
        ),
        axis=1,
    )

    return pd.concat(
        [
            output.reset_index(drop=True),
            row_metrics.reset_index(drop=True),
        ],
        axis=1,
    )



In [32]:

if RUN_ZERO_SHOT:
    zero_shot_predictions = transcribe_dataset(
        test_raw_ds,
        test_df,
        zero_shot_model,
        batch_size=ZERO_SHOT_BATCH_SIZE,
        description="Zero-shot Google MedASR",
    )

    zero_shot_metrics = {
        "model": MODEL_NAME,
        "setting": "zero-shot",
        **compute_overall_metrics(
            zero_shot_predictions
        ),
    }

    zero_shot_predictions.to_csv(
        OUTPUT_ROOT
        / "medasr_zero_shot_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )

    pd.DataFrame(
        [zero_shot_metrics]
    ).to_csv(
        OUTPUT_ROOT
        / "medasr_zero_shot_summary.csv",
        index=False,
    )

    display(
        pd.DataFrame([zero_shot_metrics])
    )

Zero-shot Google MedASR:   0%|          | 0/139 [00:00<?, ?batch/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv1d(


,model,setting,wer,cer,mwer,medical_precision,medical_recall,medical_f1,medical_accuracy,medical_tp,medical_fp,medical_fn,n
0,google/medasr,zero-shot,0.497532,0.277178,0.513837,0.979336,0.544916,0.70022,0.061628,9384,198,7837,2223


## 8. Training preprocessing and data collator

In [ ]:

def get_tokenizer(processor_object):
    if hasattr(processor_object, "tokenizer"):
        return processor_object.tokenizer
    return processor_object


def _remove_only_batch_dimension(values, name: str):
    """Remove a leading batch dimension without deleting the time axis."""
    array = np.asarray(values)

    # A processor called on one waveform may return either:
    #   [1, time, feature] / [1, time]  -> remove batch dimension
    #   [time, feature] / [time]        -> already unbatched; keep unchanged
    if array.ndim >= 3 and array.shape[0] == 1:
        array = array[0]
    elif array.ndim == 2 and array.shape[0] == 1 and name == "input_values":
        array = array[0]

    if array.ndim == 0 or array.shape[0] <= 1:
        raise ValueError(
            f"{name} lost its time dimension: shape={array.shape}. "
            "Do not blindly index processor output with [0]."
        )

    return array.astype(np.float32, copy=False)


def prepare_dataset(
    dataset: Dataset,
    processor_object,
) -> Dataset:
    tokenizer = get_tokenizer(processor_object)

    def preprocess(example):
        audio = load_audio_16k(example["audio"])

        # return_tensors='np' makes the batch dimension explicit and prevents
        # accidental removal of MedASR's acoustic time dimension.
        processed = processor_object(
            audio["array"],
            sampling_rate=audio["sampling_rate"],
            return_tensors="np",
        )

        input_key = get_model_input_key(processed)
        acoustic = _remove_only_batch_dimension(
            processed[input_key],
            input_key,
        )

        # CTC labels must not contain automatically added BOS/EOS tokens.
        label_ids = tokenizer(
            normalize_text(example["reference"]),
            add_special_tokens=False,
        ).input_ids

        if not label_ids:
            raise ValueError(
                f"Empty transcript after tokenization: {example['reference']!r}"
            )

        example[input_key] = acoustic
        example["labels"] = label_ids
        example["input_length"] = int(acoustic.shape[0])
        example["label_length"] = int(len(label_ids))
        return example

    prepared = dataset.map(
        preprocess,
        remove_columns=dataset.column_names,
        desc="Preparing Google MedASR dataset",
    )

    # Fail before a long training run if features or labels are malformed.
    sample = prepared[0]
    print(
        "Prepared sample:",
        {k: (np.asarray(v).shape if k in ("input_features", "input_values") else v)
         for k, v in sample.items() if k != "labels"},
        "label_length=", len(sample["labels"]),
    )
    return prepared


In [ ]:

@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[Dict[str, Any]],
    ) -> Dict[str, torch.Tensor]:
        if "input_features" in features[0]:
            input_name = "input_features"
        elif "input_values" in features[0]:
            input_name = "input_values"
        else:
            raise KeyError(
                "Features contain neither 'input_features' nor 'input_values'. "
                f"Available keys: {list(features[0].keys())}"
            )

        input_features = [
            {input_name: feature[input_name]}
            for feature in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        if (labels >= 0).sum().item() == 0:
            raise ValueError("All CTC labels were masked to -100.")

        batch["labels"] = labels
        return batch


## 9. Epoch metrics callback

In [ ]:
class EpochMetricsCallback(TrainerCallback):
    def __init__(self, trial_dir):
        self.trial_dir = Path(trial_dir)
        self.trial_dir.mkdir(
            parents=True,
            exist_ok=True,
        )
        self.rows = []
        self.latest_train_loss = np.nan

    def on_log(
        self,
        args,
        state,
        control,
        logs=None,
        **kwargs,
    ):
        logs = logs or {}

        if "loss" in logs:
            self.latest_train_loss = float(
                logs["loss"]
            )

    def on_evaluate(
        self,
        args,
        state,
        control,
        metrics=None,
        **kwargs,
    ):
        metrics = metrics or {}

        row = {
            "epoch": (
                float(state.epoch)
                if state.epoch is not None
                else np.nan
            ),
            "global_step": int(
                state.global_step
            ),
            "train_loss": self.latest_train_loss,
            "eval_loss": float(
                metrics.get(
                    "eval_loss",
                    np.nan,
                )
            ),
            "eval_wer": float(
                metrics.get(
                    "eval_wer",
                    np.nan,
                )
            ),
            "eval_cer": float(
                metrics.get(
                    "eval_cer",
                    np.nan,
                )
            ),
        }

        self.rows.append(row)

        pd.DataFrame(
            self.rows
        ).to_csv(
            self.trial_dir
            / "epoch_metrics.csv",
            index=False,
        )

        print(
            f"Epoch {row['epoch']:.0f} | "
            f"train loss={row['train_loss']:.4f} | "
            f"validation loss={row['eval_loss']:.4f} | "
            f"validation WER={row['eval_wer']:.4f} | "
            f"validation CER={row['eval_cer']:.4f}"
        )


def compute_trainer_metrics(
    evaluation_prediction,
) -> Dict[str, float]:
    predictions = (
        evaluation_prediction.predictions[0]
        if isinstance(evaluation_prediction.predictions, tuple)
        else evaluation_prediction.predictions
    )

    predicted_ids = np.argmax(predictions, axis=-1)
    label_ids = evaluation_prediction.label_ids.copy()
    tokenizer = get_tokenizer(processor)
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    predicted_text = processor.batch_decode(predicted_ids)
    # References are ordinary token sequences, not CTC frame sequences.
    # Therefore repeated label tokens must NOT be collapsed.
    try:
        reference_text = tokenizer.batch_decode(
            label_ids,
            skip_special_tokens=True,
            group_tokens=False,
        )
    except TypeError:
        reference_text = tokenizer.batch_decode(
            label_ids,
            skip_special_tokens=True,
        )

    predicted_text = [normalize_text(text) for text in predicted_text]
    reference_text = [normalize_text(text) for text in reference_text]

    valid = [
        (reference, prediction)
        for reference, prediction in zip(reference_text, predicted_text)
        if reference.strip()
    ]
    if not valid:
        raise ValueError("No non-empty validation references were decoded.")

    references, hypotheses = map(list, zip(*valid))
    return {
        "wer": wer(references, hypotheses),
        "cer": cer(references, hypotheses),
    }


## 10. Architecture-specific hyperparameter tuning

In [ ]:
@dataclass(frozen=True)
class TrialConfig:
    learning_rate: float
    epochs: int
    weight_decay: float
    effective_batch_size: int


trial_configs = [
    TrialConfig(
        learning_rate=learning_rate,
        epochs=epochs,
        weight_decay=weight_decay,
        effective_batch_size=EFFECTIVE_BATCH_SIZE,
    )
    for learning_rate in LEARNING_RATES
    for epochs in EPOCH_OPTIONS
    for weight_decay in WEIGHT_DECAYS
]

display(
    pd.DataFrame(
        [asdict(config) for config in trial_configs]
    )
)

,learning_rate,epochs,weight_decay,effective_batch_size
0,0.00001,10,0.01,32


### CTC preprocessing fix

The original preprocessing unconditionally selected `processed[input_key][0]`. For MedASR this can select the first **time frame**, rather than remove a batch dimension. The resulting one-frame acoustic sequence cannot align with normal transcripts, so CTC treats the examples as impossible: training loss becomes `0`, evaluation loss becomes `NaN`, and decoding returns empty hypotheses (WER/CER = `1.0`).

This revision preserves the acoustic time axis, removes BOS/EOS from CTC targets, checks that labels are not fully masked, decodes reference labels without CTC repeat-collapsing, disables unstable FP16 training, and performs a finite-loss preflight test before starting trials.


In [ ]:
def train_hyperparameter_trials(
    train_dataset,
    validation_dataset,
    output_dir,
):
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    prepared_train = prepare_dataset(
        train_dataset,
        processor,
    )

    prepared_validation = prepare_dataset(
        validation_dataset,
        processor,
    )

    collator = DataCollatorCTCWithPadding(
        processor=processor,
        padding=True,
    )


    # Preflight: a single batch must have a real time axis, usable labels,
    # and a finite non-zero CTC loss before hyperparameter trials begin.
    probe_features = [prepared_train[i] for i in range(min(2, len(prepared_train)))]
    probe_batch = collator(probe_features)
    print("Preflight batch shapes:", {k: tuple(v.shape) for k, v in probe_batch.items()})
    print("Usable label tokens:", int((probe_batch["labels"] >= 0).sum().item()))

    probe_model = AutoModelForCTC.from_pretrained(MODEL_NAME).to(DEVICE)
    probe_inputs = {k: v.to(DEVICE) for k, v in probe_batch.items()}
    probe_model.eval()
    with torch.no_grad():
        probe_output = probe_model(**probe_inputs)
    probe_loss = float(probe_output.loss.detach().cpu())
    print("Preflight CTC loss:", probe_loss)
    if not np.isfinite(probe_loss) or probe_loss <= 0.0:
        raise RuntimeError(
            "Invalid CTC preflight loss. Check acoustic feature shape, label "
            "lengths, tokenizer special tokens, and transcript/audio alignment."
        )
    del probe_model, probe_inputs, probe_output
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    tuning_rows = []
    best_wer = float("inf")
    selected_model_dir = None

    for trial_index, config in enumerate(
        trial_configs,
        start=1,
    ):
        trial_dir = (
            output_dir
            / f"trial_{trial_index:02d}"
        )

        trial_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        print(
            "\nStarting trial",
            trial_index,
            asdict(config),
        )

        model = AutoModelForCTC.from_pretrained(
            MODEL_NAME
        )

        gradient_accumulation_steps = max(
            1,
            config.effective_batch_size
            // TRAIN_BATCH_SIZE,
        )

        training_args = TrainingArguments(
            output_dir=str(trial_dir),

            learning_rate=config.learning_rate,
            weight_decay=config.weight_decay,
            num_train_epochs=config.epochs,

            per_device_train_batch_size=(
                TRAIN_BATCH_SIZE
            ),
            per_device_eval_batch_size=(
                EVAL_BATCH_SIZE
            ),

            gradient_accumulation_steps=(
                gradient_accumulation_steps
            ),

            eval_strategy="epoch",
            save_strategy="epoch",

            logging_strategy="steps",
            logging_steps=25,

            load_best_model_at_end=True,
            metric_for_best_model="wer",
            greater_is_better=False,

            save_total_limit=1,

            fp16=False,
            bf16=(
                torch.cuda.is_available()
                and torch.cuda.is_bf16_supported()
            ),

            report_to="none",

            seed=SEED,
            data_seed=SEED,

            remove_unused_columns=False,
        )

        epoch_callback = EpochMetricsCallback(
            trial_dir
        )

        trainer = Trainer(
            model=model,
            args=training_args,

            train_dataset=prepared_train,
            eval_dataset=prepared_validation,

            data_collator=collator,
            compute_metrics=compute_trainer_metrics,

            processing_class=processor,

            callbacks=[
                epoch_callback,
                EarlyStoppingCallback(
                    early_stopping_patience=(
                        EARLY_STOPPING_PATIENCE
                    )
                ),
            ],
        )

        start_time = time.time()

        train_result = trainer.train()
        evaluation = trainer.evaluate()

        runtime = time.time() - start_time

        row = {
            "trial": trial_index,
            **asdict(config),

            "eval_loss": evaluation.get(
                "eval_loss",
                np.nan,
            ),

            "eval_wer": evaluation.get(
                "eval_wer",
                np.nan,
            ),

            "eval_cer": evaluation.get(
                "eval_cer",
                np.nan,
            ),

            "train_runtime": (
                train_result.metrics.get(
                    "train_runtime",
                    np.nan,
                )
            ),

            "total_runtime_sec": runtime,

            "best_checkpoint": (
                trainer.state.best_model_checkpoint
            ),
        }

        tuning_rows.append(row)

        pd.DataFrame(
            tuning_rows
        ).to_csv(
            output_dir
            / "medasr_tuning_results_partial.csv",
            index=False,
        )

        if row["eval_wer"] < best_wer:
            best_wer = row["eval_wer"]

            selected_model_dir = (
                output_dir
                / "selected_model"
            )

            trainer.save_model(
                selected_model_dir
            )

            processor.save_pretrained(
                selected_model_dir
            )

            with (
                selected_model_dir
                / "selected_trial.json"
            ).open(
                "w",
                encoding="utf-8",
            ) as handle:

                json.dump(
                    row,
                    handle,
                    indent=2,
                    default=str,
                )

        del trainer
        del model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    tuning_results = pd.DataFrame(
        tuning_rows
    ).sort_values(
        "eval_wer"
    )

    tuning_results.to_csv(
        output_dir
        / "medasr_tuning_results.csv",
        index=False,
    )

    if selected_model_dir is None:
        raise RuntimeError(
            "No model was selected during tuning."
        )

    selected_model = (
        AutoModelForCTC.from_pretrained(
            selected_model_dir
        )
    )

    return (
        selected_model,
        tuning_results,
        selected_model_dir,
    )


## 11. Run fine-tuning

In [ ]:
selected_model = None
tuning_results = None
selected_model_dir = None

if RUN_TRAINING:
    (
        selected_model,
        tuning_results,
        selected_model_dir,
    ) = train_hyperparameter_trials(
        train_raw_ds,
        val_raw_ds,
        OUTPUT_ROOT / "tuning",
    )

    display(tuning_results)
    print(
        "Selected model directory:",
        selected_model_dir,
    )

Preparing Google MedASR dataset:   0%|          | 0/2977 [00:00<?, ? examples/s]

Prepared sample: {'input_features': (4712, 128), 'input_length': 4712, 'label_length': 229} label_length= 229


Preparing Google MedASR dataset:   0%|          | 0/999 [00:00<?, ? examples/s]

Prepared sample: {'input_features': (1097, 128), 'input_length': 1097, 'label_length': 58} label_length= 58
Preflight batch shapes: {'input_features': (2, 4712, 128), 'attention_mask': (2, 4712), 'labels': (2, 229)}
Usable label tokens: 383


Loading weights:   0%|          | 0/368 [00:00<?, ?it/s]

...

Training Loss,Validation Loss,Epoch,Wer,Cer
74.309976,2.120546,10,0.541451,0.288114


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/368 [00:00<?, ?it/s]

,trial,learning_rate,epochs,weight_decay,effective_batch_size,eval_loss,eval_wer,eval_cer,train_runtime,total_runtime_sec,best_checkpoint
0,1,0.00001,10,0.01,32,2.120546,0.541451,0.288114,9448.9575,9649.567308,/content/drive/MyDrive/Brock_University/pidgin...


Selected model directory: /content/drive/MyDrive/Brock_University/pidgin_dataset/Final_Dataset/reviewer_revision_outputs/medasr/tuning/selected_model


## 12. Fine-tuned test evaluation

In [ ]:
fine_tuned_predictions = None
fine_tuned_metrics = None

if RUN_TRAINING:
    selected_model = (
        selected_model.float().to(DEVICE)
    )

    fine_tuned_predictions = transcribe_dataset(
        test_raw_ds,
        test_df,
        selected_model,
        batch_size=EVAL_BATCH_SIZE,
        description="Fine-tuned Google MedASR",
    )

    fine_tuned_metrics = {
        "model": MODEL_NAME,
        "setting": "fine-tuned",
        **compute_overall_metrics(
            fine_tuned_predictions
        ),
    }

    fine_tuned_predictions.to_csv(
        OUTPUT_ROOT
        / "medasr_finetuned_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )

    pd.DataFrame(
        [fine_tuned_metrics]
    ).to_csv(
        OUTPUT_ROOT
        / "medasr_finetuned_summary.csv",
        index=False,
    )

    display(
        pd.DataFrame([fine_tuned_metrics])
    )

Fine-tuned Google MedASR:   0%|          | 0/1494 [00:00<?, ?batch/s]

,model,setting,wer,cer,mwer,medical_precision,medical_recall,medical_f1,medical_accuracy,medical_tp,medical_fp,medical_fn,n
0,google/medasr,fine-tuned,0.537277,0.289391,0.402602,0.985012,0.602162,0.747413,0.336011,3286,50,2171,1494


## 13. Speaker-level and code-switch analyses

In [ ]:
def grouped_metrics(
    predictions: pd.DataFrame,
    group_column: str,
) -> pd.DataFrame:
    rows = []

    for group_value, group in predictions.groupby(
        group_column,
        dropna=False,
    ):
        rows.append(
            {
                group_column: group_value,
                **compute_overall_metrics(group),
            }
        )

    return pd.DataFrame(rows)


if fine_tuned_predictions is not None:
    speaker_metrics = grouped_metrics(
        fine_tuned_predictions,
        "speaker_id",
    )

    speaker_metrics.to_csv(
        OUTPUT_ROOT
        / "medasr_speaker_metrics.csv",
        index=False,
    )

    display(speaker_metrics)

    if (
        "code_switch_category"
        in fine_tuned_predictions.columns
    ):
        code_switch_metrics = grouped_metrics(
            fine_tuned_predictions,
            "code_switch_category",
        )

        code_switch_metrics.to_csv(
            OUTPUT_ROOT
            / "medasr_code_switch_metrics.csv",
            index=False,
        )

        display(code_switch_metrics)

,speaker_id,wer,cer,mwer,medical_precision,medical_recall,medical_f1,medical_accuracy,medical_tp,medical_fp,medical_fn,n
0,Speaker_1,0.588533,0.325937,0.457746,0.985323,0.545504,0.702232,0.296593,1007,15,839,499
1,Speaker_4,0.560084,0.305425,0.453042,0.981061,0.552828,0.707167,0.256000,1036,20,838,500
2,Speaker_8,0.461138,0.235635,0.289580,0.988076,0.715602,0.830050,0.456566,1243,15,494,495


,code_switch_category,wer,cer,mwer,medical_precision,medical_recall,medical_f1,medical_accuracy,medical_tp,medical_fp,medical_fn,n
0,Low Code-Switch,0.525127,0.283022,0.398591,0.986177,0.605966,0.750674,0.314680,2925,41,1902,1233
1,Moderate Code-Switch,0.597121,0.321271,0.426910,0.974860,0.579734,0.727083,0.440945,349,9,253,254
2,No Code-Switch,0.685185,0.426782,0.571429,1.000000,0.428571,0.600000,0.285714,12,0,16,7


## 14. Export aligned manifest and run configuration

In [ ]:
manifest_columns = [
    "sample_id",
    "speaker_id",
    "split",
    "audio_path",
    "reference",
]

for optional_column in (
    "audio_length",
    "code_switch_category",
    "cmi",
    "english_ratio",
    "pidgin_ratio",
):
    if optional_column in df.columns:
        manifest_columns.append(
            optional_column
        )

df[manifest_columns].to_csv(
    OUTPUT_ROOT
    / "medasr_evaluation_manifest.csv",
    index=False,
)

run_config = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "device": DEVICE,
    "metadata_path": str(METADATA_PATH),
    "audio_root": str(AUDIO_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "split_manifest_path": str(
        SPLIT_MANIFEST_PATH
    ),
    "validated_lexicon_path": (
        str(VALIDATED_LEXICON_PATH)
        if VALIDATED_LEXICON_PATH is not None
        else None
    ),
    "test_size": TEST_SIZE,
    "validation_size_within_remaining": (
        VAL_SIZE_WITHIN_REMAINING
    ),
    "learning_rates": LEARNING_RATES,
    "epoch_options": EPOCH_OPTIONS,
    "weight_decays": WEIGHT_DECAYS,
    "effective_batch_size": (
        EFFECTIVE_BATCH_SIZE
    ),
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "early_stopping_patience": (
        EARLY_STOPPING_PATIENCE
    ),
    "run_zero_shot": RUN_ZERO_SHOT,
    "run_training": RUN_TRAINING,
}

with (
    OUTPUT_ROOT / "run_config.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        run_config,
        handle,
        indent=2,
    )

print("All outputs saved to:", OUTPUT_ROOT)

All outputs saved to: /content/drive/MyDrive/Brock_University/pidgin_dataset/Final_Dataset/reviewer_revision_outputs/medasr
